In [ ]:
# Лабораторна робота №2
# Частина 1 — Завантаження даних VHI

import pandas as pd
import urllib.request
import os
from datetime import datetime
from io import StringIO

: 

In [ ]:
# Завантаження даних VHI для всіх областей України

def download_vhi_data(save_dir="vhi_data"):
    # Створюємо папку якщо не існує
    os.makedirs(save_dir, exist_ok=True)
    
    for province_id in range(1, 28):  # 27 областей
        # Перевіряємо чи вже є файл для цієї області
        existing_files = [f for f in os.listdir(save_dir) if f"province_{province_id}_" in f]
        if existing_files:
            print(f"Область {province_id} вже завантажена, пропускаємо")
            continue
        
        # Формуємо URL
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        # Додаємо дату та час до імені файлу
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"province_{province_id}_{timestamp}.csv"
        filepath = os.path.join(save_dir, filename)
        
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Завантажено область {province_id}")
        except Exception as e:
            print(f"Помилка для області {province_id}: {e}")

download_vhi_data()

Завантажено область 1
Завантажено область 2
Завантажено область 3
Завантажено область 4
Завантажено область 5
Завантажено область 6
Завантажено область 7
Завантажено область 8
Завантажено область 9
Завантажено область 10
Завантажено область 11
Завантажено область 12
Завантажено область 13
Завантажено область 14
Завантажено область 15
Завантажено область 16
Завантажено область 17
Завантажено область 18
Завантажено область 19
Завантажено область 20
Завантажено область 21
Завантажено область 22
Завантажено область 23
Завантажено область 24
Завантажено область 25
Завантажено область 26
Завантажено область 27


In [ ]:
def read_vhi_data(save_dir="vhi_data"):
    frames = []
    
    for filename in os.listdir(save_dir):
        filepath = os.path.join(save_dir, filename)
        province_id = int(filename.split("_")[1])
        
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        # Прибираємо перші 2 рядки (заголовок і назви стовпців)
        # Прибираємо HTML теги і порожні рядки
        clean_lines = []
        for line in lines[2:]:
            line = line.strip()
            line = line.replace('<tt><pre>', '').replace('</pre></tt>', '')
            if line and not line.startswith('<'):
                # Прибираємо кому в кінці
                line = line.rstrip(',')
                clean_lines.append(line)
        
        # Створюємо DataFrame
        data = '\n'.join(clean_lines)
        df = pd.read_csv(StringIO(data), header=None, 
                        names=['year','week','SMN','SMT','VCI','TCI','VHI'],
                        skipinitialspace=True)
        
        df['province_id'] = province_id
        frames.append(df)
    
    result = pd.concat(frames, ignore_index=True)
    return result

df = read_vhi_data()
df.head(10)

In [7]:
# Data Cleaning
def clean_df(df):
    # Видаляємо непотрібні стовпці
    df = df.drop(columns=['SMN', 'SMT'], errors='ignore')
    
    # Перетворюємо типи
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['week'] = pd.to_numeric(df['week'], errors='coerce').astype('Int64')
    df['VCI'] = pd.to_numeric(df['VCI'], errors='coerce')
    df['TCI'] = pd.to_numeric(df['TCI'], errors='coerce')
    df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')
    
    # Видаляємо рядки з VHI = -1 або NaN
    df = df[df['VHI'] != -1]
    df = df.dropna(subset=['VHI'])
    
    return df

df = clean_df(df)
df.head()

,year,week,VCI,TCI,VHI,province_id
0,1982,1,51.11,48.78,49.95,10
1,1982,2,55.89,38.20,47.04,10
2,1982,3,57.30,32.69,44.99,10
3,1982,4,53.96,28.62,41.29,10
4,1982,5,46.87,28.57,37.72,10


In [9]:
# Заміна індексів областей (з англійської на українську абетку)

# Відповідність: NOAA province_id -> українська назва та новий індекс
province_mapping = {
    1: ('Черкаська', 1),
    2: ('Чернігівська', 2),
    3: ('Чернівецька', 3),
    4: ('Кримська', 4),
    5: ('Дніпропетровська', 5),
    6: ('Донецька', 6),
    7: ('Івано-Франківська', 7),
    8: ('Харківська', 8),
    9: ('Херсонська', 9),
    10: ('Хмельницька', 10),
    11: ('Київська', 11),
    12: ('Кіровоградська', 12),
    13: ('Луганська', 13),
    14: ('Львівська', 14),
    15: ('Миколаївська', 15),
    16: ('Одеська', 16),
    17: ('Полтавська', 17),
    18: ('Рівненська', 18),
    19: ('Сумська', 19),
    20: ('Тернопільська', 20),
    21: ('Закарпатська', 21),
    22: ('Вінницька', 22),
    23: ('Волинська', 23),
    24: ('Запорізька', 24),
    25: ('Житомирська', 25),
    26: ('Київська міська', 26),
    27: ('Севастопільська', 27),
}

# Додаємо стовпці з назвою та новим індексом області
df['province_name'] = df['province_id'].map(lambda x: province_mapping[x][0])
df['province_ua_id'] = df['province_id'].map(lambda x: province_mapping[x][1])

df.head()

,year,week,VCI,TCI,VHI,province_id,province_name,province_ua_id
0,1982,1,51.11,48.78,49.95,10,Хмельницька,10
1,1982,2,55.89,38.20,47.04,10,Хмельницька,10
2,1982,3,57.30,32.69,44.99,10,Хмельницька,10
3,1982,4,53.96,28.62,41.29,10,Хмельницька,10
4,1982,5,46.87,28.57,37.72,10,Хмельницька,10


In [14]:
# Функція 1 — VHI для області за вказаний рік
def vhi_by_region_year(df, province_ua_id, year):
    result = df[(df['province_ua_id'] == province_ua_id) & (df['year'] == year)]
    return result[['week', 'VHI']]

# Тест
print("VHI для Хмельницької (id=10) за 2000 рік:")
print(vhi_by_region_year(df, 10, 2000))

VHI для Хмельницької (id=10) за 2000 рік:
     week    VHI
936     1  30.96
937     2  32.40
938     3  34.26
939     4  36.86
940     5  37.50
941     6  36.27
942     7  35.69
943     8  37.18
944     9  40.80
945    10  43.00
946    11  43.01
947    12  45.33
948    13  47.26
949    14  48.35
950    15  50.09
951    16  54.24
952    17  59.31
953    18  64.13
954    19  65.00
955    20  64.35
956    21  60.95
957    22  55.25
958    23  50.15
959    24  46.80
960    25  46.61
961    26  46.86
962    27  46.08
963    28  46.22
964    29  49.70
965    30  50.85
966    31  51.24
967    32  52.31
968    33  51.46
969    34  52.58
970    35  55.55
971    36  55.58
972    37  56.31
973    38  56.47
974    39  53.94
975    40  47.14
976    41  40.59
977    42  36.74
978    43  32.23
979    44  26.58
980    45  21.90
981    46  20.97
982    47  22.85
983    48  25.17
984    49  27.76
985    50  29.01
986    51  30.24
987    52  32.87


In [15]:
# Функція 2 — VHI за діапазон років для вказаних областей
def vhi_by_regions_years(df, province_ids, year_start, year_end):
    result = df[
        (df['province_ua_id'].isin(province_ids)) & 
        (df['year'] >= year_start) & 
        (df['year'] <= year_end)
    ]
    return result[['year', 'week', 'VHI', 'province_name']]

# Тест
print("VHI для областей 10, 11 за 2000-2005:")
print(vhi_by_regions_years(df, [10, 11], 2000, 2005).head(10))

VHI для областей 10, 11 за 2000-2005:
     year  week    VHI province_name
936  2000     1  30.96   Хмельницька
937  2000     2  32.40   Хмельницька
938  2000     3  34.26   Хмельницька
939  2000     4  36.86   Хмельницька
940  2000     5  37.50   Хмельницька
941  2000     6  36.27   Хмельницька
942  2000     7  35.69   Хмельницька
943  2000     8  37.18   Хмельницька
944  2000     9  40.80   Хмельницька
945  2000    10  43.00   Хмельницька


In [13]:
# Функція 3 — Екстремуми, середнє, медіана
def vhi_stats(df, province_ids, year_start, year_end):
    data = vhi_by_regions_years(df, province_ids, year_start, year_end)
    print(f"Мінімум VHI: {data['VHI'].min()}")
    print(f"Максимум VHI: {data['VHI'].max()}")
    print(f"Середнє VHI:  {data['VHI'].mean():.2f}")
    print(f"Медіана VHI:  {data['VHI'].median()}")

# Тест
print("Статистика для областей 10, 11 за 2000-2005:")
vhi_stats(df, [10, 11], 2000, 2005)

Статистика для областей 10, 11 за 2000-2005:
Мінімум VHI: 10.6
Максимум VHI: 80.88
Середнє VHI:  51.36
Медіана VHI:  51.965
